# **Stage 2 Data Preparation, Harmonization, and Feature Engineering**

### 1. Environment Setup

In [31]:
# ======================================
# Load packages
# ======================================
from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import gender_guesser.detector as gender


# ======================================
# Utilities for notebook execution
# ======================================
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

### 2. Operational Tenant Dataset

In [32]:
# ======================================
# Load data
# ======================================
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "tenant_nov.csv"

df = pd.read_csv(DATA_PATH, sep=";")

In [33]:
# ======================================
# Sanity checks
# ======================================
print("Shape:", df.shape)

print("\nPreview:")
display(df.head(3))

print("\nColumn types:")
print(df.dtypes.value_counts())

print("\nTop missing values (%):")
print(
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (564, 16)

Preview:


,Vorname,Nachname,Mietvertrag,Einheit,Wohnung,Lage,Objekt,Buchungspartner,Buchungspartner ID,Einzug,Auszug,E-Mail,Telefon,Geburtsdatum,Nationalität,Beruf
0,NaN,The Village Psw GmbH & Co.KG,R12017C201768XW,CU.31.4,CU.31,"1005,00 Euro",CU44,NaN,NaN,01.06.2025,31.05.2026,NaN,NaN,NaN,NaN,NaN
1,Aaditya,Jhinjharia,R8550C2091506W,B.02.1,B.02,"1020,00 Euro",Aufgang B,NaN,NaN,01.08.2025,30.11.2025,aadityajhinjharia@gmail.com,8588953709,18.10.2005,IND,Student
2,aairah abidhussain,syed,R9003C2015726W,D.54.1,D.54,"1110,00 Euro",Aufgang D,NaN,NaN,10.06.2025,30.06.2026,airusyed@gmail.com,'+919867281455,22.11.2006,IND,Student



Column types:
object     14
float64     2
Name: count, dtype: int64

Top missing values (%):
Buchungspartner       1.000000
Buchungspartner ID    1.000000
Nationalität          0.095745
Telefon               0.085106
Geburtsdatum          0.058511
Beruf                 0.054965
E-Mail                0.007092
Vorname               0.001773
Nachname              0.001773
Mietvertrag           0.000000
dtype: float64

Duplicate rows: 0


In [34]:
# ======================================
# Remove Management Placeholder Entries
# ======================================

before = len(df)

mask_management = (
    df["Nachname"]
    .str.strip()
    .str.lower()
    != "the village psw gmbh & co.kg"
)

df = df[mask_management].copy()

after = len(df)

print("Rows removed (management placeholders):", before - after)

Rows removed (management placeholders): 1


#### 2.1 Translation of Variables (German to English)

In [35]:
# ======================================
# Column renaming & structural cleanup
# ======================================

rename_map = {
    "Vorname": "FirstName",
    "Nachname": "LastName",
    "Mietvertrag": "LeaseContract",
    "Einheit": "Unit",
    "Wohnung": "Apartment",
    "Lage": "Location",
    "Objekt": "Building",
    "Buchungspartner": "BookingPartner",
    "Buchungspartner ID": "BookingPartnerID",
    "Einzug": "MoveInDate",
    "Auszug": "MoveOutDate",
    "E-Mail": "Email",
    "Telefon": "Phone",
    "Geburtsdatum": "DateOfBirth",
    "Nationalität": "Nationality",
    "Beruf": "Occupation"
}

df.rename(columns=rename_map, inplace=True)

drop_cols = [
    "LastName",
    "LeaseContract",
    "Location",
    "Building",
    "BookingPartner",
    "BookingPartnerID",
    "Email",
    "Phone"
]

df.drop(columns=drop_cols, errors="ignore", inplace=True)
df.columns

Index(['FirstName', 'Unit', 'Apartment', 'MoveInDate', 'MoveOutDate',
       'DateOfBirth', 'Nationality', 'Occupation'],
      dtype='object')

In [36]:
# ======================================
# Standardize Occupation Labels (DE → EN)
# ======================================

occupation_map = {
    "Student": "Student",
    "Selbstständig": "Self-employed",
    "Angestellt": "Employed",
    "Arbeitssuchend": "Job-seeking",
    "Ausbildung": "Apprentice / Trainee",
    "Beamter": "Civil Servant",
    "Rentner": "Retired"
}

df["Occupation"] = (
    df["Occupation"]
    .str.strip()
    .replace(occupation_map)
)

In [37]:
# ======================================
# Standardize Nationality Codes (ISO → Country Name)
# ======================================

nationality_map = {
    "IND": "India",
    "DEU": "Germany",
    "EGY": "Egypt",
    "MAR": "Morocco",
    "TZA": "Tanzania",
    "YEM": "Yemen",
    "FRA": "France",
    "PAK": "Pakistan",
    "LBN": "Lebanon",
    "ESP": "Spain",
    "KAZ": "Kazakhstan",
    "ROU": "Romania",
    "ITA": "Italy",
    "MEX": "Mexico",
    "COL": "Colombia",
    "SRB": "Serbia",
    "POL": "Poland",
    "BRA": "Brazil",
    "IRN": "Iran",
    "NLD": "Netherlands",
    "UKR": "Ukraine",
    "USA": "United States",
    "LTU": "Lithuania",
    "RUS": "Russia",
    "BIH": "Bosnia and Herzegovina",
    "THA": "Thailand",
    "ARG": "Argentina",
    "NGA": "Nigeria",
    "KOR": "South Korea",
    "HUN": "Hungary",
    "CHL": "Chile",
    "TUR": "Turkey",
    "CHN": "China",
    "TWN": "Taiwan",
    "PRT": "Portugal",
    "GBR": "United Kingdom",
    "IDN": "Indonesia",
    "AUT": "Austria",
    "LUX": "Luxembourg",
    "HND": "Honduras",
    "KEN": "Kenya",
    "BGR": "Bulgaria",
    "KGZ": "Kyrgyzstan",
    "MDA": "Moldova",
    "AFG": "Afghanistan",
    "ZAF": "South Africa",
    "AZE": "Azerbaijan",
    "GRC": "Greece",
    "PRY": "Paraguay",
    "MMR": "Myanmar",
    "SYR": "Syria",
    "TJK": "Tajikistan",
    "BLR": "Belarus",
    "PER": "Peru",
    "GTM": "Guatemala",
    "CMR": "Cameroon",
    "ECU": "Ecuador",
    "LKA": "Sri Lanka",
    "IRQ": "Iraq",
    "CAN": "Canada",
    "MYS": "Malaysia",
    "EUR": "Europe (unspecified)",
    "BEL": "Belgium",
    "CHE": "Switzerland",
    "PHL": "Philippines",
    "LVA": "Latvia",
    "CRI": "Costa Rica",
    "MKD": "North Macedonia",
    "SVK": "Slovakia",
    "URY": "Uruguay",
    "BGD": "Bangladesh",
    "AUS": "Australia",
    "TUN": "Tunisia",
    "KHM": "Cambodia",
    "ZWE": "Zimbabwe",
    "NAM": "Namibia",
    "IRL": "Ireland",
    "JAM": "Jamaica",
    "MOZ": "Mozambique",
    "NPL": "Nepal",
    "JPN": "Japan",
    "ERI": "Eritrea",
    "PSX": "Palestine (unspecified)",
    "CZE": "Czechia",
    "ALB": "Albania",
    "SWE": "Sweden"
}

df["Nationality"] = (
    df["Nationality"]
    .replace(nationality_map)
)

In [38]:
# ======================================
# Handle Missing Categorical Values
# ======================================

categorical_cols = ["Nationality", "Occupation"] # these are not used as predictors but rather for descriptive stats, so we can just label missing as "Unknown"

df[categorical_cols] = df[categorical_cols].fillna("Unknown")


# ======================================
# Verification
# ======================================

for col in categorical_cols:
    print(f"{col} – Missing:", df[col].isna().sum())
    print(f"{col} – Unique values:", df[col].nunique())
    print("-" * 40)

Nationality – Missing: 0
Nationality – Unique values: 87
----------------------------------------
Occupation – Missing: 0
Occupation – Unique values: 7
----------------------------------------


#### 2.2 Temporal Feature Construction

In [39]:
# ======================================
# Date Parsing & Inspection
# ======================================

date_cols = ["MoveInDate", "MoveOutDate", "DateOfBirth"]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        dayfirst=True,
        errors="coerce"
    )


# ======================================
# Verification
# ======================================

for col in date_cols:
    print(f"{col}")
    print("dtype:", df[col].dtype)
    print("Min:", df[col].min(), "| Max:", df[col].max())
    print("NaT count:", df[col].isna().sum())
    print("-" * 40)

MoveInDate
dtype: datetime64[ns]
Min: 2024-11-29 00:00:00 | Max: 2025-11-10 00:00:00
NaT count: 0
----------------------------------------
MoveOutDate
dtype: datetime64[ns]
Min: 2025-11-30 00:00:00 | Max: 2026-11-30 00:00:00
NaT count: 0
----------------------------------------
DateOfBirth
dtype: datetime64[ns]
Min: 1970-03-11 00:00:00 | Max: 2008-06-06 00:00:00
NaT count: 32
----------------------------------------


In [40]:
# ======================================
# Derived Temporal Features
# ======================================

ref_date = pd.Timestamp("2025-11-25")  # Survey/Operational Tenant Data reference date


# ---- Age ----
df["Age"] = ref_date.year - df["DateOfBirth"].dt.year

birthday_passed = (
    (df["DateOfBirth"].dt.month < ref_date.month) |
    (
        (df["DateOfBirth"].dt.month == ref_date.month) &
        (df["DateOfBirth"].dt.day <= ref_date.day)
    )
)

df.loc[~birthday_passed, "Age"] -= 1


# ---- Length of stay (days since move-in) ----
df["LengthOfStay_days"] = (
    (ref_date - df["MoveInDate"])
    .dt.days
    .clip(lower=0)
)


# ---- Remaining stay (days until move-out) ----
df["RemainingStay_days"] = (
    (df["MoveOutDate"] - ref_date)
    .dt.days
    .clip(lower=0)
)


# Preview
df[[
    "Unit",
    "MoveInDate",
    "MoveOutDate",
    "Age",
    "LengthOfStay_days",
    "RemainingStay_days"
]].head()

,Unit,MoveInDate,MoveOutDate,Age,LengthOfStay_days,RemainingStay_days
1,B.02.1,2025-08-01,2025-11-30,20.0,116,5
2,D.54.1,2025-06-10,2026-06-30,19.0,168,217
3,FR.71.5,2025-01-01,2025-12-30,25.0,328,35
4,A.26.3,2025-07-12,2026-07-31,21.0,136,248
5,D.31.1,2025-07-01,2026-06-30,20.0,147,217


In [41]:
# ======================================
# Handle Missing Age Values (Median Imputation)
# ======================================

median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)

print("Missing Age after imputation:", df["Age"].isna().sum())
print("Median age used for imputation:", median_age)

Missing Age after imputation: 0
Median age used for imputation: 24.0


#### 2.3 Additional Transformations (Apartment Size and Gender)

In [42]:
# ======================================
# Add Apartment Size Feature (max room number per apartment)
# ======================================

def add_apartment_size(df, unit_col="Unit"):
    df = df.copy()

    # Extract apartment code (e.g., A.12 or MO.71)
    df["apartment_tmp"] = df[unit_col].str.extract(r"^([A-Z]{1,2}\.\d{2})\.\d+$")

    # Extract room number (last digit after final dot)
    df["room_number_tmp"] = (
        df[unit_col]
        .str.extract(r"\.(\d+)$")
        .astype(float)
    )

    # Compute max room number per apartment
    df["ApartmentSize"] = (
        df.groupby("apartment_tmp")["room_number_tmp"]
        .transform("max")
        .astype("Int64")
    )

    df.drop(columns=["apartment_tmp", "room_number_tmp"], inplace=True)

    return df


df = add_apartment_size(df)

In [43]:
# Obtain gender column through gender guesser

d = gender.Detector(case_sensitive=False)

def predict_gender_from_name(name):
    # Missing → unknown
    if pd.isna(name):
        return "unknown"
    
    s = str(name).strip()
    if not s:
        return "unknown"
    
    # Split on spaces and hyphens
    tokens = re.split(r"[\s\-]+", s)
    tokens = [t.lower() for t in tokens if t]

    raw_guess = "unknown"

    for token in tokens:
        g = d.get_gender(token)
        # Take the first non-unknown guess
        if g != "unknown":
            raw_guess = g
            break

    # Map raw gender-guesser outputs to clean labels
    if raw_guess in ["male", "mostly_male"]:
        return "male"
    if raw_guess in ["female", "mostly_female"]:
        return "female"
    # "andy" (androgynous) and "unknown" both become unknown
    return "unknown"

In [44]:
# ======================================
# Apply Gender Inference & Inspect Results
# ======================================

df["Gender"] = df["FirstName"].apply(predict_gender_from_name)


# Distribution
gender_counts = df["Gender"].value_counts(dropna=False)
print(gender_counts)

Gender
male       262
unknown    169
female     132
Name: count, dtype: int64


In [45]:
# ======================================
# Reduce Dataset for Survey Merge
# ======================================

df_reduced = df[[
    "Unit",                 # exact rental unit (e.g., A.51.2)
    "Apartment",            # shared apartment code (e.g., A.51)
    "Gender",               # inferred from first name
    "Age",
    "Occupation",
    "Nationality",
    "LengthOfStay_days",
    "RemainingStay_days",
    "ApartmentSize"         # rooms in apartment
]].copy()

df_reduced.head()

,Unit,Apartment,Gender,Age,Occupation,Nationality,LengthOfStay_days,RemainingStay_days,ApartmentSize
1,B.02.1,B.02,unknown,20.0,Student,India,116,5,4
2,D.54.1,D.54,unknown,19.0,Student,India,168,217,3
3,FR.71.5,FR.71,male,25.0,Employed,Germany,328,35,5
4,A.26.3,A.26,male,21.0,Employed,Egypt,136,248,4
5,D.31.1,D.31,male,20.0,Student,Morocco,147,217,2


### 3. Real Tenant Compatibility Responses (Limited Coverage)

In [46]:
# ======================================
# Load Survey Responses
# ======================================
SURVEY_PATH = PROJECT_ROOT / "data" / "raw" / "survey_responses.csv"

responses_raw = pd.read_csv(SURVEY_PATH)

# Drop non-response header rows (Qualtrics export)
responses = (
    responses_raw
    .iloc[2:]
    .reset_index(drop=True)
)

responses.head()

,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,...,Q31,Q31_7_TEXT,Q32,Q34,Q44_1,Q44_2,Q44_3,Q44_4,Q44_5,Q44_6
0,2025-11-19 12:59:59,2025-11-19 13:02:51,Survey Preview,NaN,100,172,True,2025-11-19 13:02:52,R_8E14HpoKC1XBF0I,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-11-20 11:06:40,2025-11-20 11:06:45,Survey Preview,NaN,100,5,True,2025-11-20 11:06:46,R_8CkgtpJwVrSILA9,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-11-20 11:55:17,2025-11-20 12:02:12,Survey Preview,NaN,100,415,True,2025-11-20 12:02:12,R_8EQFIxsWvcDWvKv,NaN,...,"Respect for boundaries and personal space,Reli...",NaN,Moderately,Very compatible,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-11-20 12:12:12,2025-11-20 12:14:29,Survey Preview,NaN,100,137,True,2025-11-20 12:14:30,R_86SrbnkpW3iML6x,NaN,...,Reliability and responsibility in shared spaces,NaN,Very,Not at all compatible,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-11-20 12:16:16,2025-11-20 12:20:38,Survey Preview,NaN,100,262,True,2025-11-20 12:20:38,R_86itPI2eKCM6bIJ,NaN,...,Being social and engaging,NaN,Slightly,Very compatible,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
# ======================================
# Filter Valid Survey Responses
# ======================================

# Remove preview runs
if "Status" in responses.columns:
    responses = responses[
        responses["Status"] != "Survey Preview"
    ]


# Keep only finished responses
if "Finished" in responses.columns:

    if responses["Finished"].dtype == "O":
        responses = responses[
            responses["Finished"]
            .astype(str)
            .str.lower()
            .isin(["1", "true", "yes"])
        ]
    else:
        responses = responses[
            responses["Finished"] == 1
        ]


# Keep only consented participants
if "Consent" in responses.columns:
    responses = responses[
        responses["Consent"]
        .str.strip()
        == "I agree to participate."
    ]


responses = responses.reset_index(drop=True)

responses.shape

(8, 55)

In [48]:
# ======================================
# Rename Survey Variables
# ======================================

# ---- IPIP Items ----
ipip_cols = {
    f"Q43_{i}": f"IPIP_{i}"
    for i in range(1, 11)
}

responses.rename(columns=ipip_cols, inplace=True)


# ---- Compatibility & Context Variables ----
rename_map = {
    "Q31": "FlatmateFactors_Priority",
    "Q32": "Compatibility_Importance",
    "Q34": "Compatibility_OverallRating",

    "Q44_1": "Compat_DailyRoutines",
    "Q44_2": "Compat_Tidiness",
    "Q44_3": "Compat_Noise",
    "Q44_4": "Compat_Interaction",
    "Q44_5": "Compat_Conflict",
    "Q44_6": "Compat_Relaxedness",

    "Occupation_7_TEXT": "Occupation_Other",
    "Gender_5_TEXT": "Gender_Other",
    "Q31_7_TEXT": "FlatmateFactors_Other",
    "RoomID": "Apartment"
}

responses.rename(columns=rename_map, inplace=True)

In [49]:
# ======================================
# Remove Metadata & Redundant Variables
# ======================================

cols_to_drop = [
    "StartDate", "EndDate", "Status", "IPAddress", "Progress",
    "Duration (in seconds)", "Finished", "RecordedDate",
    "ResponseId", "RecipientLastName", "RecipientFirstName",
    "RecipientEmail", "ExternalReference",
    "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage",

    # Not needed (handled via tenant dataset)
    "Gender_Other",
    "Occupation",
    "Occupation_Other",
    "Consent",
    "FlatmateFactors_Other",
    "Age",
    "Gender"
]

responses = responses.drop(
    columns=[c for c in cols_to_drop if c in responses.columns]
)

responses.columns

Index(['Apartment', 'IPIP_1', 'IPIP_2', 'IPIP_3', 'IPIP_4', 'IPIP_5', 'IPIP_6',
       'IPIP_7', 'IPIP_8', 'IPIP_9', 'IPIP_10', 'Sleep_schedule',
       'Noise_sensitivity', 'Cleanliness', 'Cleanliness_2', 'Cooking_at_home',
       'Guests_over', 'Smoking', 'Alcohol', 'Vibe', 'Chores', 'Conflict',
       'FlatmateFactors_Priority', 'Compatibility_Importance',
       'Compatibility_OverallRating', 'Compat_DailyRoutines',
       'Compat_Tidiness', 'Compat_Noise', 'Compat_Interaction',
       'Compat_Conflict', 'Compat_Relaxedness'],
      dtype='object')

In [50]:
# ======================================
# Inspect Survey Sample Size
# ======================================

print("Total rows in raw export:", len(responses_raw))

# Properly remove Qualtrics metadata rows (first 3 rows)
responses = responses_raw.iloc[3:].reset_index(drop=True)

print("Rows after removing metadata:", len(responses))


# Apply minimal validity filter (finished + consent only)
valid_mask = pd.Series(True, index=responses.index)

if "Finished" in responses.columns:
    valid_mask &= responses["Finished"].astype(str).str.lower().isin(["true", "1"])

if "Consent" in responses.columns:
    valid_mask &= responses["Consent"].str.contains("agree", case=False, na=False)

responses_valid = responses[valid_mask].copy()

print("Valid survey responses:", len(responses_valid))
print("Invalid / filtered out:", len(responses) - len(responses_valid))

Total rows in raw export: 26
Rows after removing metadata: 23
Valid survey responses: 18
Invalid / filtered out: 5


### 4. Synthetic Tenant Compatibility Responses

In [51]:
# ======================================
# Load Synthetic Survey Data
# ======================================
SYNTHETIC_PATH = PROJECT_ROOT / "data" / "synthetic" / "synthetic_survey_data.csv"

syn_responses = pd.read_csv(SYNTHETIC_PATH)

print("Number of synthetic responses:", len(syn_responses))
syn_responses.head()

Number of synthetic responses: 563


,Apartment,IPIP_1,IPIP_2,IPIP_3,IPIP_4,IPIP_5,IPIP_6,IPIP_7,IPIP_8,IPIP_9,...,Conflict,FlatmateFactors_Priority,Compatibility_Importance,Compatibility_OverallRating,Compat_DailyRoutines,Compat_Tidiness,Compat_Noise,Compat_Interaction,Compat_Conflict,Compat_Relaxedness
0,B.02,Neither agree nor disagree,Strongly disagree,Neither agree nor disagree,Strongly agree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Disagree,...,Try to solve problems through discussion,Respect for boundaries,Moderately,Slightly compatible,Agree,Agree,Agree,Strongly agree,Disagree,Agree
1,D.54,Agree,Agree,Neither agree nor disagree,Neither agree nor disagree,Disagree,Disagree,Agree,Agree,Disagree,...,Try to solve problems through discussion,Similar daily routines,Moderately,Moderately compatible,Agree,Strongly disagree,Strongly disagree,Agree,Agree,Agree
2,FR.71,Neither agree nor disagree,Neither agree nor disagree,Disagree,Neither agree nor disagree,Disagree,Neither agree nor disagree,Neither agree nor disagree,Disagree,Agree,...,"Be direct, even if uncomfortable",Being social and engaging,Very,Slightly compatible,Strongly agree,Agree,Neither agree nor disagree,Strongly agree,Neither agree nor disagree,Neither agree nor disagree
3,A.26,Agree,Disagree,Neither agree nor disagree,Disagree,Neither agree nor disagree,Agree,Disagree,Neither agree nor disagree,Disagree,...,Try to solve problems through discussion,Shared interests / personality fit,Moderately,Very compatible,Neither agree nor disagree,Strongly agree,Neither agree nor disagree,Neither agree nor disagree,Strongly agree,Disagree
4,D.31,Agree,Disagree,Disagree,Disagree,Neither agree nor disagree,Strongly disagree,Disagree,Disagree,Neither agree nor disagree,...,Try to solve problems through discussion,Being social and engaging,Not at all,Not at all compatible,Strongly disagree,Strongly disagree,Disagree,Strongly agree,Agree,Agree


### 5. Data Integration and Anonymization

In [52]:
# ======================================
# Merge Simulated Responses with Tenant Data
# ======================================

# Ensure consistent ordering
syn_responses = (
    syn_responses
    .sort_values("Apartment")
    .reset_index(drop=True)
)

df_reduced = (
    df_reduced
    .sort_values("Apartment")
    .reset_index(drop=True)
)


# Create within-apartment row index
syn_responses["apt_row"] = (
    syn_responses
    .groupby("Apartment")
    .cumcount()
)

df_reduced["apt_row"] = (
    df_reduced
    .groupby("Apartment")
    .cumcount()
)


# Merge on Apartment + within-apartment index
merged_df = pd.merge(
    syn_responses,
    df_reduced,
    on=["Apartment", "apt_row"],
    how="inner",
    validate="one_to_one"
)


# Remove helper column
merged_df.drop(columns="apt_row", inplace=True)

merged_df.head()

,Apartment,IPIP_1,IPIP_2,IPIP_3,IPIP_4,IPIP_5,IPIP_6,IPIP_7,IPIP_8,IPIP_9,...,Compat_Conflict,Compat_Relaxedness,Unit,Gender,Age,Occupation,Nationality,LengthOfStay_days,RemainingStay_days,ApartmentSize
0,A.01,Agree,Disagree,Strongly agree,Agree,Neither agree nor disagree,Disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,...,Agree,Disagree,A.01.2,male,25.0,Employed,Poland,97,279,2
1,A.01,Disagree,Disagree,Disagree,Neither agree nor disagree,Disagree,Agree,Neither agree nor disagree,Disagree,Neither agree nor disagree,...,Strongly disagree,Agree,A.01.1,unknown,19.0,Student,South Korea,55,309,2
2,A.02,Disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Neither agree nor disagree,Disagree,Neither agree nor disagree,...,Strongly agree,Strongly agree,A.02.2,male,28.0,Employed,Kyrgyzstan,55,309,2
3,A.02,Disagree,Agree,Disagree,Agree,Neither agree nor disagree,Neither agree nor disagree,Strongly agree,Agree,Agree,...,Neither agree nor disagree,Neither agree nor disagree,A.02.1,male,25.0,Student,Unknown,24,340,2
4,A.03,Disagree,Disagree,Neither agree nor disagree,Disagree,Neither agree nor disagree,Neither agree nor disagree,Agree,Agree,Neither agree nor disagree,...,Disagree,Agree,A.03.3,male,20.0,Student,Spain,55,309,4


In [53]:
# ======================================
# Check which apartment ID's have different format (MO41 instead of MO.41)
# ======================================
print("Unique apartments in merged dataset:", merged_df["Apartment"].nunique())
print("Unique apartment values:", merged_df["Apartment"].unique())


Unique apartments in merged dataset: 167
Unique apartment values: ['A.01' 'A.02' 'A.03' 'A.04' 'A.05' 'A.06' 'A.11' 'A.12' 'A.13' 'A.14'
 'A.15' 'A.16' 'A.21' 'A.22' 'A.23' 'A.24' 'A.25' 'A.26' 'A.31' 'A.32'
 'A.33' 'A.34' 'A.35' 'A.36' 'A.41' 'A.42' 'A.43' 'A.44' 'A.45' 'A.46'
 'A.51' 'A.52' 'A.53' 'A.54' 'A.55' 'A.56' 'A.61' 'A.62' 'A.63' 'B.01'
 'B.02' 'B.03' 'B.04' 'B.11' 'B.12' 'B.13' 'B.14' 'B.21' 'B.22' 'B.23'
 'B.24' 'B.31' 'B.32' 'B.33' 'B.34' 'B.41' 'B.42' 'B.43' 'B.44' 'B.51'
 'B.52' 'B.53' 'B.54' 'B.61' 'B.62' 'B.63' 'C.01' 'C.02' 'C.03' 'C.04'
 'C.11' 'C.12' 'C.13' 'C.14' 'C.21' 'C.22' 'C.23' 'C.24' 'C.31' 'C.32'
 'C.33' 'C.34' 'C.41' 'C.42' 'C.43' 'C.44' 'C.51' 'C.52' 'C.53' 'C.54'
 'C.61' 'C.62' 'CU.11' 'CU.12' 'CU.21' 'CU.22' 'CU.31' 'CU.32' 'CU.41'
 'CU.42' 'CU.51' 'D.01' 'D.02' 'D.03' 'D.04' 'D.11' 'D.12' 'D.13' 'D.14'
 'D.21' 'D.22' 'D.23' 'D.24' 'D.31' 'D.32' 'D.33' 'D.34' 'D.41' 'D.42'
 'D.43' 'D.44' 'D.51' 'D.52' 'D.53' 'D.54' 'D.61' 'D.62' 'E.01' 'E.02'
 'E.11' '

In [54]:
# ======================================
# Normalize and anonymize Unit / Apartment codes in merged_df
# ======================================

work_df = merged_df.copy()

# --------------------------------------
# 1) Normalize Apartment / Unit formats
# Fix cases like MO51 -> MO.51 and MO511 -> MO.51.1
# --------------------------------------

def normalize_apartment_code(x):
    if pd.isna(x):
        return x
    x = str(x).strip()

    # already correct: e.g. MO.51
    if re.fullmatch(r"[A-Za-z]{1,2}\.\d{2}", x):
        return x

    # fix: MO51 -> MO.51
    m = re.fullmatch(r"([A-Za-z]{1,2})(\d{2})", x)
    if m:
        return f"{m.group(1)}.{m.group(2)}"

    return x


def normalize_unit_code(x):
    if pd.isna(x):
        return x
    x = str(x).strip()

    # already correct: e.g. MO.51.1
    if re.fullmatch(r"[A-Za-z]{1,2}\.\d{2}\.\d{1,2}", x):
        return x

    # fix: MO511 -> MO.51.1
    m = re.fullmatch(r"([A-Za-z]{1,2})(\d{2})(\d{1,2})", x)
    if m:
        return f"{m.group(1)}.{m.group(2)}.{m.group(3)}"

    return x

work_df["Apartment"] = work_df["Apartment"].apply(normalize_apartment_code)
work_df["Unit"] = work_df["Unit"].apply(normalize_unit_code)

print("Normalized Apartment examples:")
display(work_df[["Apartment"]].drop_duplicates().head(20))

print("Normalized Unit examples:")
display(work_df[["Unit"]].drop_duplicates().head(20))


# --------------------------------------
# 2) Pseudonymize Unit / Apartment
# Preserve:
# - Apartment as Unit prefix
# - original room number
# - missing-room logic
# --------------------------------------

def int_to_alpha(n: int) -> str:
    n += 1
    code = ""
    while n > 0:
        n, rem = divmod(n - 1, 26)
        code = chr(65 + rem) + code
    return code

apt_pattern = r'^(?P<raw_building>[A-Za-z]+)\.(?P<raw_apartment>\d{2})$'
unit_pattern = r'^(?P<raw_building>[A-Za-z]+)\.(?P<raw_apartment>\d{2})\.(?P<raw_room>\d{1,2})$'

apt_parts = work_df["Apartment"].astype(str).str.strip().str.extract(apt_pattern)
unit_parts = work_df["Unit"].astype(str).str.strip().str.extract(unit_pattern)

# --------------------------------------
# 3) Validate normalized format
# --------------------------------------

bad_apts = work_df.loc[apt_parts.isna().any(axis=1), "Apartment"].dropna().unique().tolist()
bad_units = work_df.loc[unit_parts.isna().any(axis=1), "Unit"].dropna().unique().tolist()

if bad_apts:
    raise ValueError(f"Unexpected Apartment format found after normalization: {bad_apts[:10]}")

if bad_units:
    raise ValueError(f"Unexpected Unit format found after normalization: {bad_units[:10]}")

# --------------------------------------
# 4) Validate Unit prefix matches Apartment
# --------------------------------------

work_df["Apartment_original"] = work_df["Apartment"].astype(str).str.strip()
work_df["Unit_original"] = work_df["Unit"].astype(str).str.strip()

unit_apartment_prefix = unit_parts["raw_building"] + "." + unit_parts["raw_apartment"]
mismatch_mask = unit_apartment_prefix != work_df["Apartment_original"]

if mismatch_mask.any():
    mismatch_rows = work_df.loc[mismatch_mask, ["Unit_original", "Apartment_original"]].head(10)
    raise ValueError(
        "Some Unit / Apartment pairs do not match after normalization.\n"
        f"{mismatch_rows}"
    )

# --------------------------------------
# 5) Build apartment mapping table
# --------------------------------------

unique_apts = (
    pd.DataFrame({"Apartment_original": work_df["Apartment_original"]})
    .drop_duplicates()
    .sort_values("Apartment_original")
    .reset_index(drop=True)
)

unique_apts[["raw_building", "raw_apartment"]] = (
    unique_apts["Apartment_original"].str.extract(apt_pattern)
)

# Building cover codes that can never match originals
unique_buildings = sorted(unique_apts["raw_building"].str.upper().unique())
building_map = {b: f"PX{int_to_alpha(i)}" for i, b in enumerate(unique_buildings)}

unique_apts["building_cover"] = unique_apts["raw_building"].str.upper().map(building_map)

# Renumber apartments sequentially within building
unique_apts = unique_apts.sort_values(["raw_building", "raw_apartment"]).reset_index(drop=True)
unique_apts["apartment_cover_no"] = (
    unique_apts.groupby("raw_building").cumcount().add(1).map(lambda x: f"{x:02d}")
)

unique_apts["Apartment_cover"] = (
    unique_apts["building_cover"] + "." + unique_apts["apartment_cover_no"]
)

apartment_map = dict(zip(unique_apts["Apartment_original"], unique_apts["Apartment_cover"]))

# --------------------------------------
# 6) Replace Apartment and rebuild Unit
# --------------------------------------

work_df["Apartment"] = work_df["Apartment_original"].map(apartment_map)
work_df["RoomNo"] = unit_parts["raw_room"].astype(str)
work_df["Unit"] = work_df["Apartment"] + "." + work_df["RoomNo"]

# --------------------------------------
# 7) Validation checks
# --------------------------------------

assert (work_df["Unit"].str.rsplit(".", n=1).str[0] == work_df["Apartment"]).all()

same_apartments = (work_df["Apartment_original"] == work_df["Apartment"]).sum()
same_units = (work_df["Unit_original"] == work_df["Unit"]).sum()

print("Unchanged apartments:", same_apartments)
print("Unchanged units:", same_units)

# --------------------------------------
# 8) Optional inspection
# --------------------------------------

mapping_preview = unique_apts[[
    "Apartment_original", "Apartment_cover", "raw_building", "raw_apartment"
]].copy()

print("\nSample apartment mapping:")
display(mapping_preview.head(10))

print("\nSample transformed rows:")
display(work_df[[
    "Unit_original", "Unit", "Apartment_original", "Apartment"
]].head(10))

# --------------------------------------
# 9) Cleanup
# --------------------------------------

work_df = work_df.drop(columns=["RoomNo"])

# drop originals from final safe dataset
merged_df_clean = work_df.drop(columns=["Unit_original", "Apartment_original"]).copy()

Normalized Apartment examples:


,Apartment
0,A.01
2,A.02
4,A.03
8,A.04
11,A.05
15,A.06
19,A.11
21,A.12
24,A.13
28,A.14


Normalized Unit examples:


,Unit
0,A.01.2
1,A.01.1
2,A.02.2
3,A.02.1
4,A.03.3
5,A.03.1
6,A.03.2
7,A.03.4
8,A.04.3
9,A.04.2


Unchanged apartments: 0
Unchanged units: 0

Sample apartment mapping:


,Apartment_original,Apartment_cover,raw_building,raw_apartment
0,A.01,PXA.01,A,01
1,A.02,PXA.02,A,02
2,A.03,PXA.03,A,03
3,A.04,PXA.04,A,04
4,A.05,PXA.05,A,05
5,A.06,PXA.06,A,06
6,A.11,PXA.07,A,11
7,A.12,PXA.08,A,12
8,A.13,PXA.09,A,13
9,A.14,PXA.10,A,14



Sample transformed rows:


,Unit_original,Unit,Apartment_original,Apartment
0,A.01.2,PXA.01.2,A.01,PXA.01
1,A.01.1,PXA.01.1,A.01,PXA.01
2,A.02.2,PXA.02.2,A.02,PXA.02
3,A.02.1,PXA.02.1,A.02,PXA.02
4,A.03.3,PXA.03.3,A.03,PXA.03
5,A.03.1,PXA.03.1,A.03,PXA.03
6,A.03.2,PXA.03.2,A.03,PXA.03
7,A.03.4,PXA.03.4,A.03,PXA.03
8,A.04.3,PXA.04.3,A.04,PXA.04
9,A.04.2,PXA.04.2,A.04,PXA.04


In [55]:
# ======================================
# Back to merged_df name
# ======================================

# Rename merged_df_clean back to merged_df
merged_df = merged_df_clean.copy()
merged_df.columns

Index(['Apartment', 'IPIP_1', 'IPIP_2', 'IPIP_3', 'IPIP_4', 'IPIP_5', 'IPIP_6',
       'IPIP_7', 'IPIP_8', 'IPIP_9', 'IPIP_10', 'Sleep_schedule',
       'Noise_sensitivity', 'Cleanliness', 'Cleanliness_2', 'Cooking_at_home',
       'Guests_over', 'Smoking', 'Alcohol', 'Vibe', 'Chores', 'Conflict',
       'FlatmateFactors_Priority', 'Compatibility_Importance',
       'Compatibility_OverallRating', 'Compat_DailyRoutines',
       'Compat_Tidiness', 'Compat_Noise', 'Compat_Interaction',
       'Compat_Conflict', 'Compat_Relaxedness', 'Unit', 'Gender', 'Age',
       'Occupation', 'Nationality', 'LengthOfStay_days', 'RemainingStay_days',
       'ApartmentSize'],
      dtype='object')

#### 5.1 Personality Feature Construction and Value Handling

In [56]:
# ======================================
# Construct Big Five Personality Scores
# ======================================

# ---- Convert Likert labels → numeric (1–5) ----
likert_to_num = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neither agree nor disagree": 3,
    "Agree": 4,
    "Strongly agree": 5
}

ipip_cols = [f"IPIP_{i}" for i in range(1, 11)]

for col in ipip_cols:
    merged_df[col] = merged_df[col].map(likert_to_num)


# ---- Reverse-coded items ----
# 3  = few artistic interests
# 4  = tend to be lazy
# 6  = get nervous easily
# 8  = find fault with others
# 10 = reserved

reverse_items = ["IPIP_3", "IPIP_4", "IPIP_6", "IPIP_8", "IPIP_10"]

for col in reverse_items:
    merged_df[col] = 6 - merged_df[col]


# ---- Big Five composites (2 items each) ----

merged_df["Emotional_Stability"] = (
    merged_df[["IPIP_1", "IPIP_6"]].mean(axis=1)
)

merged_df["Extraversion"] = (
    merged_df[["IPIP_2", "IPIP_10"]].mean(axis=1)
)

merged_df["Openness"] = (
    merged_df[["IPIP_3", "IPIP_7"]].mean(axis=1)
)

merged_df["Conscientiousness"] = (
    merged_df[["IPIP_4", "IPIP_9"]].mean(axis=1)
)

merged_df["Agreeableness"] = (
    merged_df[["IPIP_5", "IPIP_8"]].mean(axis=1)
)

#### 5.2 Ordinal Encoding of Lifestyle, Behaviour, and Compatibility Variables

In [57]:
# ======================================
# Ordinal Encoding (Survey → Numeric)
# ======================================
# Convention: higher = more / stronger / more frequent

ordinal_maps = {
    # Sensitivity / importance
    "Noise_sensitivity": {
        "Very insensitive": 1,
        "Somewhat insensitive": 2,
        "Neutral": 3,
        "Somewhat sensitive": 4,
        "Very sensitive": 5,
    },
    "Cleanliness": {
        "Not at all important": 1,
        "Slightly important": 2,
        "Moderately important": 3,
        "Very important": 4,
        "Extremely important": 5,
    },

    # Lifestyle / routine
    "Sleep_schedule": {
        "Before 22:00": 1,
        "22:00-00:00": 2,
        "00:00-02:00": 3,
        "After 02:00": 4,
        "Prefer not to say": np.nan,
    },

    # Frequency behaviours
    "Cleanliness_2": {
        "Rarely": 1,
        "Sometimes": 2,
        "Often": 3,
        "Most of the time": 4,
        "Always": 5,
    },
    "Cooking_at_home": {
        "Never": 1,
        "Rarely": 2,
        "Once per week": 3,
        "Several times per week": 4,
        "Every day": 5,
    },
    "Guests_over": {
        "Never": 1,
        "Rarely": 2,
        "A few times per month": 3,
        "Once per week": 4,
        "Several times per week": 5,
    },
    "Alcohol": {
        "Never": 1,
        "Rarely": 2,
        "Sometimes": 3,
        "Often": 4,
        "Very often": 5,
    },

    # Social preference intensity
    "Vibe": {
        "Quiet & private": 1,
        "Mostly private": 2,
        "Balanced": 3,
        "Social & lively": 4,
        "Very social": 5,
    },
    "Chores": {
        "Avoid them": 1,
        "Do them when asked": 2,
        "Do my fair share": 3,
        "Often take initiative": 4,
        "Take responsibility for most": 5,
    },
    "Conflict": {
        "Avoid confrontation": 1,
        "Compromise quickly": 2,
        "Try to solve problems through discussion": 3,
        "Be direct, even if uncomfortable": 4,
        "Prefer not to say": np.nan,
    },

    # Perceived compatibility
    "Compatibility_Importance": {
        "Not at all": 1,
        "Slightly": 2,
        "Moderately": 3,
        "Very": 4,
        "Extremely": 5,
    },
    "Compatibility_OverallRating": {
        "Not at all compatible": 1,
        "Slightly compatible": 2,
        "Moderately compatible": 3,
        "Very compatible": 4,
        "Extremely compatible": 5,
    },

    # Binary
    "Smoking": {
        "Non-smoker": 0,
        "Smoker": 1,
    }
}


# ======================================
# Apply mappings
# ======================================

for col, mapping in ordinal_maps.items():
    if col in merged_df.columns:
        merged_df[f"{col}_num"] = merged_df[col].map(mapping)


# Apply likert mapping to sub-compatibility ratings as well
compat_cols = [f"Compat_{factor}" for factor in [
    "DailyRoutines", "Tidiness", "Noise", "Interaction",
    "Conflict", "Relaxedness"
]]

for col in compat_cols:
    if col in merged_df.columns:
        merged_df[f"{col}_num"] = merged_df[col].map(likert_to_num)


# ======================================
# Sanity check: missing after encoding
# ======================================

encoded_cols = [f"{c}_num" for c in list(ordinal_maps.keys()) + compat_cols if c in merged_df.columns]

missing_rate = (
    merged_df[encoded_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print(missing_rate)

Conflict_num                       0.008881
Compatibility_Importance_num       0.000000
Compat_Conflict_num                0.000000
Compat_Interaction_num             0.000000
Compat_Noise_num                   0.000000
Compat_Tidiness_num                0.000000
Compat_DailyRoutines_num           0.000000
Smoking_num                        0.000000
Compatibility_OverallRating_num    0.000000
Noise_sensitivity_num              0.000000
Cleanliness_num                    0.000000
Chores_num                         0.000000
Vibe_num                           0.000000
Alcohol_num                        0.000000
Guests_over_num                    0.000000
Cooking_at_home_num                0.000000
Cleanliness_2_num                  0.000000
Sleep_schedule_num                 0.000000
Compat_Relaxedness_num             0.000000
dtype: float64


#### 5.3 Final Preparation for CSV Export

In [58]:
# ======================================
# Drop IPIP items and reorder columns
# ======================================

# drop raw IPIP item columns
ipip_cols = [f"IPIP_{i}" for i in range(1, 11)]
merged_df = merged_df.drop(columns=ipip_cols, errors="ignore")

# desired column order
ordered_cols = [
    # identifiers / demographics
    "Unit",
    "Apartment",
    "Age",
    "Gender",
    "Nationality",
    
    # tenancy
    "RemainingStay_days",
    "LengthOfStay_days",
    
    # Big Five traits
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Emotional_Stability",
    "Openness",
    
    # lifestyle / behaviour (raw)
    "Sleep_schedule",
    "Noise_sensitivity",
    "Cleanliness",
    "Cleanliness_2",
    "Cooking_at_home",
    "Guests_over",
    "Smoking",
    "Alcohol",
    "Vibe",
    "Chores",
    "Conflict",
    
    # matching preference
    "FlatmateFactors_Priority",
    "Compatibility_Importance",
    
    # apartment-level compatibility reference (raw)
    "Compatibility_OverallRating",
    "Compat_DailyRoutines",
    "Compat_Tidiness",
    "Compat_Noise",
    "Compat_Interaction",
    "Compat_Conflict",
    "Compat_Relaxedness",
    
    # numeric versions
    "Sleep_schedule_num",
    "Noise_sensitivity_num",
    "Cleanliness_num",
    "Cleanliness_2_num",
    "Cooking_at_home_num",
    "Guests_over_num",
    "Smoking_num",
    "Alcohol_num",
    "Vibe_num",
    "Chores_num",
    "Conflict_num",
    "Compatibility_Importance_num",
    "Compatibility_OverallRating_num",
    "Compat_DailyRoutines_num",
    "Compat_Tidiness_num",
    "Compat_Noise_num",
    "Compat_Interaction_num",
    "Compat_Conflict_num",
    "Compat_Relaxedness_num",
]

# keep only columns that actually exist
ordered_cols = [col for col in ordered_cols if col in merged_df.columns]

# append any remaining columns not explicitly listed
remaining_cols = [col for col in merged_df.columns if col not in ordered_cols]
merged_df = merged_df[ordered_cols + remaining_cols]

# set Unit as index
if "Unit" in merged_df.columns:
    merged_df = merged_df.set_index("Unit")

merged_df.head()

,Apartment,Age,Gender,Nationality,RemainingStay_days,LengthOfStay_days,Extraversion,Agreeableness,Conscientiousness,Emotional_Stability,...,Compatibility_Importance_num,Compatibility_OverallRating_num,Compat_DailyRoutines_num,Compat_Tidiness_num,Compat_Noise_num,Compat_Interaction_num,Compat_Conflict_num,Compat_Relaxedness_num,Occupation,ApartmentSize
Unit,,,,,,,,,,,,,,,,,,,,,
PXA.01.2,PXA.01,25.0,male,Poland,279,97,2.5,3.0,2.5,4.0,...,3,4,4,3,3,1,4,2,Employed,2
PXA.01.1,PXA.01,19.0,unknown,South Korea,309,55,2.5,3.0,3.0,2.0,...,5,4,4,1,3,4,1,4,Student,2
PXA.02.2,PXA.02,28.0,male,Kyrgyzstan,309,55,2.5,3.5,3.0,2.5,...,1,1,2,4,4,2,5,5,Employed,2
PXA.02.1,PXA.02,25.0,male,Unknown,340,24,3.5,2.5,3.0,2.5,...,4,4,4,1,5,3,3,3,Student,2
PXA.03.3,PXA.03,20.0,male,Spain,309,55,3.0,2.5,3.5,2.5,...,2,2,4,4,1,2,2,4,Student,4


In [59]:
# ======================================
# Save cleaned dataset for merging with survey data
# ======================================
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "merged_df.csv"

merged_df.to_csv(OUTPUT_PATH, index=True)